# Telco Customer Churn Baseline Experiment

This notebook establishes a reasonable experimentation-stage churn prediction baseline before adding production-style MLOps components. The goal is to create an honest baseline workflow that can later be refactored into reusable training code, tested, served through an API, containerised and monitored.

This stage does not include API serving, Docker, CI, MLflow tracking, prediction logging or drift detection.

## 1. Data Loading

The Telco Customer Churn CSV file is expected at:

`data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv`

The dataset is intentionally not committed to the repository. Download the file separately and keep the original filename.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATASET_FILENAME = "WA_Fn-UseC_-Telco-Customer-Churn.csv"


def find_project_root(start_path: Path) -> Path:
    """Walk upward from start_path to find the repository root."""
    start_path = start_path.resolve()
    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        if (candidate / "README.md").is_file() and (candidate / "data" / "raw").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from the repository "
        "root or from the notebooks/ directory. The project root must contain "
        "README.md and data/raw/."
    )


CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_WORKING_DIRECTORY)
DATASET_PATH = PROJECT_ROOT / "data" / "raw" / DATASET_FILENAME
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Current working directory: {CURRENT_WORKING_DIRECTORY}")
print(f"Resolved project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        "Download the Telco Customer Churn CSV file named "
        "WA_Fn-UseC_-Telco-Customer-Churn.csv and place it at "
        "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
    )

df = pd.read_csv(DATASET_PATH)
print(f"Loaded dataset from {DATASET_PATH}")

Current working directory: D:\Warwick\Portfolio Plan\portfolio-projects\mlops-customer-churn-platform\notebooks
Resolved project root: D:\Warwick\Portfolio Plan\portfolio-projects\mlops-customer-churn-platform
Dataset path: D:\Warwick\Portfolio Plan\portfolio-projects\mlops-customer-churn-platform\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv
Dataset exists: True
Loaded dataset from D:\Warwick\Portfolio Plan\portfolio-projects\mlops-customer-churn-platform\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv


## 2. Basic EDA

This section checks the dataset shape, columns, target balance, missing values, blank `TotalCharges` values and basic numeric summaries. Values are computed from the loaded CSV rather than hard-coded.

In [2]:
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
column_overview = pd.DataFrame({
    "column": df.columns,
    "dtype": [df[col].dtype for col in df.columns],
})
display(column_overview)

,column,dtype
0,customerID,object
1,gender,object
2,SeniorCitizen,int64
3,Partner,object
4,Dependents,object
5,tenure,int64
6,PhoneService,object
7,MultipleLines,object
8,InternetService,object
9,OnlineSecurity,object


In [4]:
target_distribution = df["Churn"].value_counts(dropna=False)
target_distribution_pct = df["Churn"].value_counts(normalize=True, dropna=False).round(4)

display(pd.DataFrame({
    "count": target_distribution,
    "proportion": target_distribution_pct,
}))

,count,proportion
Churn,,
No,5174,0.7346
Yes,1869,0.2654


In [5]:
missing_values = df.isna().sum().sort_values(ascending=False)
display(missing_values[missing_values > 0].to_frame("missing_count"))

blank_total_charges = df["TotalCharges"].astype(str).str.strip().eq("").sum()
print(f"Blank TotalCharges values before conversion: {blank_total_charges}")

,missing_count


Blank TotalCharges values before conversion: 11


In [6]:
display(df.describe(include=[np.number]).T)

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


## 3. Data Cleaning

The `customerID` column is an identifier and is dropped before modelling. `TotalCharges` is converted to numeric with `errors="coerce"` because the raw CSV stores it as an object/string column and may contain blank values. Missing `TotalCharges` values are handled later by median imputation inside the preprocessing pipeline.

The target variable `Churn` is converted from `Yes`/`No` to `1`/`0`, where churn is the positive class.

In [7]:
model_df = df.copy()

model_df["TotalCharges"] = pd.to_numeric(model_df["TotalCharges"], errors="coerce")
total_charges_missing_after_conversion = int(model_df["TotalCharges"].isna().sum())
print(
    "TotalCharges missing values after numeric conversion:",
    total_charges_missing_after_conversion,
)

model_df["Churn"] = model_df["Churn"].map({"No": 0, "Yes": 1})

if model_df["Churn"].isna().any():
    raise ValueError("Unexpected target values found in Churn column.")

model_df = model_df.drop(columns=["customerID"])
model_df.head()

TotalCharges missing values after numeric conversion: 11


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 4. Feature Split

Expected numerical features are usually `SeniorCitizen`, `tenure`, `MonthlyCharges` and `TotalCharges`. Categorical features are the remaining non-target columns after dropping `customerID`.

In [8]:
target_col = "Churn"
X = model_df.drop(columns=[target_col])
y = model_df[target_col].astype(int)

numerical_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]
categorical_features = [
    col for col in X.columns if col not in numerical_features
]

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)
print("Number of features after dropping ID and target:", X.shape[1])

Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Number of features after dropping ID and target: 19


## 5. Train/Test Split

The split uses `test_size=0.2`, `random_state=42` and `stratify=y` so the churn class distribution is represented in both train and test sets.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
display(pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "test_count": y_test.value_counts().sort_index(),
}))

Train size: 5634
Test size: 1409


,train_count,test_count
Churn,,
0,4139,1035
1,1495,374


## 6. Preprocessing Pipeline

The preprocessing pipeline uses scikit-learn `ColumnTransformer`:

- numeric features: median imputation and standard scaling
- categorical features: most-frequent imputation and one-hot encoding with unknown categories ignored

In [10]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

## 7. Baseline Models

Two baselines are used:

- `DummyClassifier`: a sanity-check baseline that predicts using the class prior distribution. It is not a meaningful churn model.
- `LogisticRegression`: the first reasonable baseline model for this tabular binary classification task.

The model comparison is intentionally simple at this stage. The project should first establish a clear baseline before moving to refactoring, serving, testing and monitoring.

In [11]:
models = {
    "DummyClassifier": DummyClassifier(strategy="prior", random_state=RANDOM_STATE),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
}

pipelines = {
    name: Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    for name, model in models.items()
}

## 8. Evaluation Metrics

Models are evaluated on the held-out test set. Churn is encoded as the positive class (`1`). Metrics include ROC-AUC where probabilities are available, accuracy, precision, recall, F1 and confusion matrix. `zero_division=0` is used for precision and recall to avoid metric errors for the dummy sanity-check model.

In [12]:
def evaluate_model(name, pipeline, X_train, X_test, y_train, y_test):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = None

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    return {
        "model": name,
        "roc_auc": None if roc_auc is None else round(float(roc_auc), 4),
        "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
        "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, y_pred, zero_division=0)), 4),
        "f1": round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
        "confusion_matrix": cm.tolist(),
    }

results = []
for model_name, pipeline in pipelines.items():
    results.append(evaluate_model(model_name, pipeline, X_train, X_test, y_train, y_test))

metrics_df = pd.DataFrame(results).set_index("model")
display(metrics_df)

,roc_auc,accuracy,precision,recall,f1,confusion_matrix
model,,,,,,
DummyClassifier,0.5000,0.7346,0.0000,0.0000,0.000,"[[1035, 0], [374, 0]]"
LogisticRegression,0.8419,0.8055,0.6572,0.5588,0.604,"[[926, 109], [165, 209]]"


## 9. Results Summary

The table above compares a sanity-check `DummyClassifier` with a simple `LogisticRegression` baseline. The dummy model establishes a minimum reference point. The logistic regression model is a reasonable first modelling baseline because it uses the same preprocessing pipeline that can later be refactored into reusable training code.

In [13]:
for result in results:
    print(result["model"])
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(np.array(result["confusion_matrix"]))
    print()

DummyClassifier
Confusion matrix [[TN, FP], [FN, TP]]:
[[1035    0]
 [ 374    0]]

LogisticRegression
Confusion matrix [[TN, FP], [FN, TP]]:
[[926 109]
 [165 209]]



## 10. Baseline Interpretation

The `LogisticRegression` baseline shows what can be achieved with a straightforward tabular ML workflow: basic cleaning, train/test split, preprocessing and a simple classifier. It is a reasonable experimentation-stage baseline because it is transparent, quick to run and easy to refactor.

The `DummyClassifier` is only a sanity check. It helps confirm that the real baseline model is being compared against a trivial reference, but it should not be interpreted as a useful churn prediction model.

At this point, the project should move toward reproducibility, reusable training and evaluation scripts, API serving and monitoring rather than heavy model tuning. The portfolio value of this project comes from demonstrating the MLOps lifecycle around a credible baseline, not from chasing the highest possible leaderboard metric.

## 11. Baseline Limitations

This is an experimentation-stage baseline. It does not include API serving, Docker packaging, automated tests, MLflow tracking, prediction logging or drift monitoring.

Model performance may not generalise to real customer populations. The Telco Customer Churn dataset is public and limited, and simulated traffic in later stages will not be equivalent to real production traffic. Any future use of churn predictions should support human review rather than automatically deciding customer treatment.

## 12. Save Metrics and Summary

If this notebook runs successfully with the real local dataset, it saves:

- `reports/baseline_metrics.json`
- `reports/baseline_summary.md`

These files are generated from the actual local run.

In [14]:
target_distribution_payload = {
    str(label): int(count)
    for label, count in df["Churn"].value_counts().sort_index().items()
}

metrics_payload = {
    "dataset_path": str(DATASET_PATH),
    "n_rows": int(df.shape[0]),
    "n_columns": int(df.shape[1]),
    "n_features_after_dropping_id_and_target": int(X.shape[1]),
    "target_distribution": target_distribution_payload,
    "total_charges_missing_after_conversion": total_charges_missing_after_conversion,
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "models": [result["model"] for result in results],
    "metrics": {result["model"]: {k: v for k, v in result.items() if k != "model"} for result in results},
    "run_date": datetime.now(timezone.utc).isoformat(),
}

metrics_path = REPORTS_DIR / "baseline_metrics.json"
summary_path = REPORTS_DIR / "baseline_summary.md"

with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

summary_metrics_df = metrics_df.drop(columns=["confusion_matrix"]).copy()
metric_columns = ["roc_auc", "accuracy", "precision", "recall", "f1"]
summary_table_lines = [
    "| model | " + " | ".join(metric_columns) + " |",
    "|---|" + "---|" * len(metric_columns),
]
for model_name, row in summary_metrics_df[metric_columns].iterrows():
    values = ["" if pd.isna(row[col]) else str(row[col]) for col in metric_columns]
    summary_table_lines.append(f"| {model_name} | " + " | ".join(values) + " |")
summary_md_table = "\n".join(summary_table_lines)

target_dist_lines = "\n".join(
    f"- {label}: {count}" for label, count in target_distribution_payload.items()
)

summary_text = f"""# Baseline Experiment Summary

## Baseline Purpose

This baseline establishes a reasonable experimentation-stage churn prediction workflow before adding production-style MLOps components.

## Dataset Summary

- Dataset path: `{DATASET_PATH}`
- Rows: {df.shape[0]}
- Columns: {df.shape[1]}
- Features after dropping `customerID` and `Churn`: {X.shape[1]}

## Target Distribution

{target_dist_lines}

## TotalCharges Cleaning Note

`TotalCharges` was converted to numeric with `pandas.to_numeric(..., errors="coerce")`. Missing values after conversion: {total_charges_missing_after_conversion}. These missing values are handled with median imputation inside the preprocessing pipeline.

## Metrics Table

{summary_md_table}

## Confusion Matrices

- DummyClassifier: {metrics_payload['metrics']['DummyClassifier']['confusion_matrix']}
- LogisticRegression: {metrics_payload['metrics']['LogisticRegression']['confusion_matrix']}

## Short Interpretation

`DummyClassifier` is a sanity-check baseline, not a meaningful churn model. `LogisticRegression` is the first reasonable baseline because it combines standard tabular preprocessing with a simple classifier that is easy to inspect and later refactor.

## Baseline Limitations

This stage does not include API serving, Docker, automated tests, MLflow tracking, prediction logging or drift monitoring. The dataset is public and limited, and results should not be treated as evidence of real production performance or real business impact.

## Next Step

Refactor the baseline logic into reusable training and evaluation scripts under `src`.
"""

summary_path.write_text(summary_text, encoding="utf-8")

print(f"Saved metrics to {metrics_path}")
print(f"Saved summary to {summary_path}")

Saved metrics to D:\Warwick\Portfolio Plan\portfolio-projects\mlops-customer-churn-platform\reports\baseline_metrics.json
Saved summary to D:\Warwick\Portfolio Plan\portfolio-projects\mlops-customer-churn-platform\reports\baseline_summary.md
